In [2]:
import pygame, math, numpy as np

LARGURA, ALTURA = 900, 650

class GoToGoalRobot:
    def __init__(self, x, y):
        self.x, self.y, self.theta = float(x), float(y), 0.0
        self.sensor_angles = [-math.pi/2, -math.pi/4, 0.0, math.pi/4, math.pi/2]
        self.sensor_range = 200.0
        self.sensor_readings = [self.sensor_range] * 5

    def cast_rays(self, obstacles):
        self.sensor_readings = []
        for beta in self.sensor_angles:
            angle = self.theta + beta
            min_dist = self.sensor_range
            for step in range(5, int(self.sensor_range), 4):
                rx = self.x + step * math.cos(angle)
                ry = self.y + step * math.sin(angle)
                if rx <= 0 or rx >= LARGURA or ry <= 0 or ry >= ALTURA:
                    min_dist = float(step)
                    break
                if any(obs.collidepoint(rx, ry) for obs in obstacles):
                    min_dist = float(step)
                    break
            self.sensor_readings.append(min_dist)

    def update_gotogoal(self, target_x, target_y):
        dx = target_x - self.x
        dy = target_y - self.y
        dist_alvo = math.hypot(dx, dy)
        
        if dist_alvo < 10.0:
            return
            
        angle_to_goal = math.atan2(dy, dx)
        erro_theta = math.atan2(math.sin(angle_to_goal - self.theta), math.cos(angle_to_goal - self.theta))
        
        v = 2.0
        w = 0.05 * erro_theta
        
        if min(self.sensor_readings) < 60.0:
            for i, leitura in enumerate(self.sensor_readings):
                if leitura < 60.0:
                    angulo_relativo = self.sensor_angles[i]
                    w += -math.copysign(1.0, angulo_relativo) * (30.0 / leitura)
                    
        self.theta += w
        self.x += v * math.cos(self.theta)
        self.y += v * math.sin(self.theta)

def main_lab5():
    pygame.init()
    screen = pygame.display.set_mode((LARGURA, ALTURA))
    robot = GoToGoalRobot(100, 100)
    obstacles = [pygame.Rect(400, 200, 100, 300)]
    alvo_x, alvo_y = 800, 325
    
    running = True
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT: running = False
            if event.type == pygame.MOUSEBUTTONDOWN:
                alvo_x, alvo_y = pygame.mouse.get_pos()
                
        robot.cast_rays(obstacles)
        robot.update_gotogoal(alvo_x, alvo_y)
        
        screen.fill((20, 24, 30))
        for obs in obstacles: pygame.draw.rect(screen, (180, 50, 50), obs)
        
        pygame.draw.circle(screen, (0, 255, 0), (alvo_x, alvo_y), 8)
        pygame.draw.circle(screen, (0, 200, 255), (int(robot.x), int(robot.y)), 16)
        
        pygame.display.flip()
        pygame.time.Clock().tick(60)
    pygame.quit()

if __name__ == "__main__": main_lab5()